# Vertical Velocity Regression Cross-Sections

This notebook builds monthly time-vertical regression sections for regional ERA5 vertical velocity (`w`) over `95-125E, -6 to 2`.

Outputs:
- simultaneous monthly regression with centered 3-month Niño3.4
- DJF-anchored regression using the JF-year convention
- `P1 = 1981-2006`, `P2 = 2007-2025`, and `Δr = P2 - P1`

DJF convention used here:
- `DJF 1981 = Dec 1980 + Jan 1981 + Feb 1981`
- the seasonal window for that target year is `Jul(y-1) ... Feb(y)`.


## 1. Imports and settings

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import xarray as xr
from scipy.stats import t as student_t

plt.rcParams.update({
    'font.family': 'Helvetica',
    'font.sans-serif': ['Helvetica', 'Arial', 'DejaVu Sans'],
    'axes.titleweight': 'regular',
    'figure.dpi': 120,
})

PROJECT_ROOT = Path('/Users/rizzie/TugasAkhir/data_processing')
W_PATH = PROJECT_ROOT / 'external/ClimateData/era5-monthly/vertical_velocity_1980-2025_1000hpa-100hpa.nc'
NINO34_PATH = PROJECT_ROOT / 'external/ClimateData/index-monthly/nino34.anom.csv'
RESULTS_DIR = PROJECT_ROOT / 'results/analisis_regresi_26-19'
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

TARGET_BOX = {
    'lon_min': 95.0,
    'lon_max': 125.0,
    'lat_min': -6.0,
    'lat_max': 2.0,
}

P1_YEARS = np.arange(1981, 2007)
P2_YEARS = np.arange(2007, 2026)
SEASON_YEARS = np.arange(1981, 2026)

MONTH_SLOTS = [
    ('Jul(0)', 7, -1),
    ('Aug(0)', 8, -1),
    ('Sep(0)', 9, -1),
    ('Oct(0)', 10, -1),
    ('Nov(0)', 11, -1),
    ('Dec(0)', 12, -1),
    ('Jan(1)', 1, 0),
    ('Feb(1)', 2, 0),
]
SLOT_POS = np.arange(len(MONTH_SLOTS))
SLOT_LABELS = [item[0] for item in MONTH_SLOTS]
PRESSURE_TICKS = [1000, 925, 850, 700, 500, 400, 300, 200, 100]
reg_levels = np.arange(-1, 1.01, 0.05)
reg_ticks = np.arange(-1, 1.01, 0.25)


def symmetric_levels(*arrays, n_levels=21, n_ticks=9):
    values = []
    for arr in arrays:
        if arr is None:
            continue
        vals = np.asarray(arr).ravel()
        vals = vals[np.isfinite(vals)]
        if vals.size:
            values.append(vals)
    if values:
        merged = np.concatenate(values)
        absmax = float(np.nanmax(np.abs(merged)))
    else:
        absmax = 1.0
    if not np.isfinite(absmax) or np.isclose(absmax, 0.0):
        absmax = 1.0
    levels = np.linspace(-absmax, absmax, n_levels)
    ticks = np.linspace(-absmax, absmax, n_ticks)
    return levels, ticks, absmax


## 2. Load data

In [ ]:
def open_w_dataset(path: Path) -> xr.Dataset:
    ds = xr.open_dataset(
        path,
        chunks={'time': 12, 'level': 1, 'lat': 181, 'lon': 360},
        decode_times=False,
    )
    ds = ds.drop_vars('step', errors='ignore')
    return xr.decode_cf(ds)


def load_nino34_series(path: Path) -> tuple[pd.Series, pd.Series]:
    df = pd.read_csv(path, parse_dates=['Date'])
    value_col = next(col for col in df.columns if col != 'Date')
    series = pd.to_numeric(df[value_col], errors='coerce').replace(-9999.0, np.nan)
    series = pd.Series(series.to_numpy(), index=df['Date'], name='nino34').sort_index()
    series = series.loc['1980-01-01':'2025-12-01'].copy()
    series_3mo = series.rolling(3, center=True, min_periods=3).mean()
    return series, series_3mo


def build_centered_3mo_running_mean(series: pd.Series) -> pd.Series:
    return series.rolling(3, center=True, min_periods=3).mean().dropna()


def monthly_anomaly(da: xr.DataArray) -> xr.DataArray:
    return da.groupby('time.month') - da.groupby('time.month').mean('time')




def standardize_series(series: pd.Series) -> pd.Series:
    series = pd.Series(series.copy(), index=series.index, name=series.name)
    mean = series.mean(skipna=True)
    std = series.std(ddof=0, skipna=True)
    if std == 0 or not np.isfinite(std):
        return series * np.nan
    return (series - mean) / std


def build_djf_index(nino_raw: pd.Series, years: np.ndarray) -> pd.Series:
    values = []
    for year in years:
        dec = pd.Timestamp(year - 1, 12, 1)
        jan = pd.Timestamp(year, 1, 1)
        feb = pd.Timestamp(year, 2, 1)
        values.append(np.nanmean([nino_raw.loc[dec], nino_raw.loc[jan], nino_raw.loc[feb]]))
    return pd.Series(values, index=years, name='nino_djf')




def standardize_da_over_dim(da: xr.DataArray, dim: str) -> xr.DataArray:
    mean = da.mean(dim, skipna=True)
    std = da.std(dim, skipna=True)
    std = xr.where(std == 0, np.nan, std)
    return (da - mean) / std


def _finite_mask(*arrays):
    mask = None
    for arr in arrays:
        current = xr.apply_ufunc(np.isfinite, arr, dask='parallelized', output_dtypes=[bool])
        mask = current if mask is None else (mask & current)
    return mask


def simple_regression_stats(field, index, sample_dim):
    field, index = xr.align(field, index, join='inner')
    valid = _finite_mask(field, index)
    x = index.where(valid)
    y = field.where(valid)

    n = valid.sum(sample_dim).astype(np.float32)
    safe_n = xr.where(n > 0, n, np.nan)

    sx = x.sum(sample_dim, skipna=True)
    sy = y.sum(sample_dim, skipna=True)
    sxx = (x * x).sum(sample_dim, skipna=True)
    syy = (y * y).sum(sample_dim, skipna=True)
    sxy = (x * y).sum(sample_dim, skipna=True)

    sxx_c = sxx - (sx ** 2 / safe_n)
    sxy_c = sxy - (sx * sy / safe_n)
    syy_c = syy - (sy ** 2 / safe_n)

    slope = sxy_c / sxx_c
    df = safe_n - 2
    sse = syy_c - (sxy_c ** 2 / sxx_c)
    mse = sse / df
    se = np.sqrt(mse / sxx_c)
    t_stat = slope / se
    p = xr.apply_ufunc(
        lambda t, dof: 2 * student_t.sf(np.abs(t), df=dof),
        t_stat,
        df,
        vectorize=True,
        dask='parallelized',
        output_dtypes=[float],
    )

    valid_stats = (n >= 3) & np.isfinite(slope) & np.isfinite(p)
    slope = slope.where(valid_stats)
    p = p.where(valid_stats)
    return slope.astype(np.float32), p.astype(np.float32), n


def pooled_ols_interaction_stats(field_past, field_recent, index_past, index_recent, sample_dim):
    field_past, index_past = xr.align(field_past, index_past, join='inner')
    field_recent, index_recent = xr.align(field_recent, index_recent, join='inner')

    valid_past = _finite_mask(field_past, index_past)
    valid_recent = _finite_mask(field_recent, index_recent)

    y0 = field_past.where(valid_past)
    x0 = index_past.where(valid_past)
    y1 = field_recent.where(valid_recent)
    x1 = index_recent.where(valid_recent)

    def _pooled_ols_1d(y0, x0, y1, x1):
        y0 = np.asarray(y0, dtype=float)
        x0 = np.asarray(x0, dtype=float)
        y1 = np.asarray(y1, dtype=float)
        x1 = np.asarray(x1, dtype=float)

        valid0 = np.isfinite(y0) & np.isfinite(x0)
        valid1 = np.isfinite(y1) & np.isfinite(x1)
        y = np.concatenate([y0[valid0], y1[valid1]])
        x = np.concatenate([x0[valid0], x1[valid1]])
        d = np.concatenate([
            np.zeros(valid0.sum(), dtype=float),
            np.ones(valid1.sum(), dtype=float),
        ])

        n = y.size
        if n < 4:
            return np.nan, np.nan, np.nan

        X = np.column_stack([np.ones(n), x, d, x * d])
        try:
            beta, *_ = np.linalg.lstsq(X, y, rcond=None)
            resid = y - X @ beta
            rss = float(np.sum(resid ** 2))
            df = float(n - X.shape[1])
            if df <= 0:
                return np.nan, np.nan, df
            xtx_inv = np.linalg.inv(X.T @ X)
            sigma2 = rss / df
            se = float(np.sqrt(sigma2 * xtx_inv[3, 3]))
            diff = float(beta[3])
            if not np.isfinite(diff) or not np.isfinite(se) or se == 0:
                return np.nan, np.nan, df
            t_stat = diff / se
            p = float(2 * student_t.sf(np.abs(t_stat), df=df))
            return diff, p, df
        except np.linalg.LinAlgError:
            return np.nan, np.nan, np.nan

    diff, p, df = xr.apply_ufunc(
        _pooled_ols_1d,
        y0,
        x0,
        y1,
        x1,
        input_core_dims=[[sample_dim], [sample_dim], [sample_dim], [sample_dim]],
        output_core_dims=[[], [], []],
        join='override',
        exclude_dims={sample_dim},
        vectorize=True,
        dask='parallelized',
        output_dtypes=[float, float, float],
        dask_gufunc_kwargs={'allow_rechunk': True},
    )
    return diff.astype(np.float32), p.astype(np.float32), df.astype(np.float32)


def season_window_date(year: int, month: int, year_shift: int) -> pd.Timestamp:
    return pd.Timestamp(year + year_shift, month, 1)


def build_window_cube(
    w_anom: xr.DataArray,
    nino_centered_3mo: pd.Series,
    nino_djf: pd.Series,
    years: np.ndarray,
) -> tuple[xr.DataArray, xr.DataArray, xr.DataArray]:
    w_rows = []
    nino_sim_rows = []
    nino_djf_rows = []

    for year in years:
        w_slots = []
        sim_slots = []
        djf_slots = []
        for _, month, year_shift in MONTH_SLOTS:
            date = season_window_date(year, month, year_shift)
            w_slots.append(w_anom.sel(time=date).values)
            sim_slots.append(float(nino_centered_3mo.loc[date]))
            djf_slots.append(float(nino_djf.loc[year]))
        w_rows.append(np.stack(w_slots, axis=0))
        nino_sim_rows.append(np.array(sim_slots, dtype=float))
        nino_djf_rows.append(np.array(djf_slots, dtype=float))

    w_cube = xr.DataArray(
        np.stack(w_rows, axis=0),
        coords={'season_year': years, 'slot': SLOT_POS, 'level': w_anom['level'].values},
        dims=('season_year', 'slot', 'level'),
        name='w_anom_cube',
    )
    nino_sim_cube = xr.DataArray(
        np.stack(nino_sim_rows, axis=0),
        coords={'season_year': years, 'slot': SLOT_POS},
        dims=('season_year', 'slot'),
        name='nino_centered_3mo',
    )
    nino_djf_cube = xr.DataArray(
        np.stack(nino_djf_rows, axis=0),
        coords={'season_year': years, 'slot': SLOT_POS},
        dims=('season_year', 'slot'),
        name='nino_djf',
    )
    return w_cube, nino_sim_cube, nino_djf_cube




def reg_by_slot_level(q_cube: xr.DataArray, nino_cube: xr.DataArray) -> tuple[xr.DataArray, xr.DataArray, xr.DataArray]:
    slope, p, n = simple_regression_stats(q_cube, nino_cube, 'season_year')
    return slope, p, n


def plot_cross_section(ax, da: xr.DataArray, title: str, cbar_label: str, sig_mask: xr.DataArray | None = None) -> None:
    mesh = ax.contourf(
        da['slot'].values,
        da['level'].values,
        da.transpose('level', 'slot').values,
        levels=reg_levels,
        cmap='bwr',
        extend='both',
    )
    ax.contour(
        da['slot'].values,
        da['level'].values,
        da.transpose('level', 'slot').values,
        levels=reg_ticks,
        colors='k',
        linewidths=0.5,
        alpha=0.45,
    )
    if sig_mask is not None:
        sig = sig_mask.transpose('level', 'slot')
        yy, xx = np.where(sig.values)
        if yy.size > 0:
            ax.scatter(
                sig['slot'].values[xx],
                sig['level'].values[yy],
                s=18,
                c='k',
                marker='.',
                linewidths=0,
                alpha=0.8,
                zorder=4,
            )
    ax.set_title(title, loc='center', fontsize=18, fontweight='bold', pad=22)
    ax.set_xticks(SLOT_POS)
    ax.set_xticklabels(SLOT_LABELS)
    ax.set_xlabel('Bulan', fontsize=17, labelpad=10)
    ax.set_ylim(float(da['level'].max()), float(da['level'].min()))
    ax.set_yticks(da['level'].values)
    ax.set_yticklabels([str(int(v)) for v in da['level'].values])
    ax.set_ylabel('Level tekanan (hPa)', fontsize=17, labelpad=12)
    ax.tick_params(axis='both', labelsize=13)
    return mesh


In [ ]:


w_ds = open_w_dataset(W_PATH)
w_box = w_ds['w'].sel(lat=slice(2, -6), lon=slice(95, 125))
lat_weights = np.cos(np.deg2rad(w_box['lat']))
w_region_mean = w_box.weighted(lat_weights).mean(('lat', 'lon'))
w_anom = monthly_anomaly(w_region_mean).load()
w_anom = standardize_da_over_dim(w_anom, 'time')

nino_raw, nino_centered_3mo = load_nino34_series(NINO34_PATH)
nino_raw = standardize_series(nino_raw)
nino_centered_3mo = build_centered_3mo_running_mean(nino_raw)
nino_djf = build_djf_index(nino_raw, SEASON_YEARS)

print('w dataset dims:', dict(w_ds.sizes))
print('regional mean dims:', dict(w_region_mean.sizes))
print('anomaly dims:', dict(w_anom.sizes))
print('nino range:', nino_raw.index.min(), 'to', nino_raw.index.max())
print('DJF years:', int(SEASON_YEARS.min()), 'to', int(SEASON_YEARS.max()))


## 3. Sanity check and regression cubes

In [ ]:
print('w_anom dims:', dict(w_anom.sizes))
print('w_anom time range:', pd.Timestamp(w_anom['time'].values[0]), 'to', pd.Timestamp(w_anom['time'].values[-1]))
print('w_anom levels:', w_anom['level'].values)

w_cube_p1, nino_sim_p1, nino_djf_p1 = build_window_cube(w_anom, nino_centered_3mo, nino_djf, P1_YEARS)
w_cube_p2, nino_sim_p2, nino_djf_p2 = build_window_cube(w_anom, nino_centered_3mo, nino_djf, P2_YEARS)

reg_sim_p1, reg_sim_p1_p, _ = reg_by_slot_level(w_cube_p1, nino_sim_p1)
reg_sim_p2, reg_sim_p2_p, _ = reg_by_slot_level(w_cube_p2, nino_sim_p2)
reg_djf_p1, reg_djf_p1_p, _ = reg_by_slot_level(w_cube_p1, nino_djf_p1)
reg_djf_p2, reg_djf_p2_p, _ = reg_by_slot_level(w_cube_p2, nino_djf_p2)

reg_sim_delta, reg_sim_delta_p, _ = pooled_ols_interaction_stats(w_cube_p1, w_cube_p2, nino_sim_p1, nino_sim_p2, 'season_year')
reg_djf_delta, reg_djf_delta_p, _ = pooled_ols_interaction_stats(w_cube_p1, w_cube_p2, nino_djf_p1, nino_djf_p2, 'season_year')

reg_levels, reg_ticks, _ = symmetric_levels(
    reg_sim_p1, reg_sim_p2, reg_sim_delta,
    reg_djf_p1, reg_djf_p2, reg_djf_delta,
)

print('simultaneous P1 shape:', reg_sim_p1.shape)
print('simultaneous P2 shape:', reg_sim_p2.shape)
print('DJF-anchored P1 shape:', reg_djf_p1.shape)
print('DJF-anchored P2 shape:', reg_djf_p2.shape)


## 4. Plot simultaneous monthly regression

In [ ]:
plot_specs = [
    (reg_sim_p1, reg_sim_p1_p <= 0.05, 'Regresi Vertical Velocity vs Nino34 - P1 1981-2006', 'Slope regresi', 'reg_verticalvelocity_simultaneous_p1.png'),
    (reg_sim_p2, reg_sim_p2_p <= 0.05, 'Regresi Vertical Velocity vs Nino34 - P2 2007-2025', 'Slope regresi', 'reg_verticalvelocity_simultaneous_p2.png'),
    (reg_sim_delta, reg_sim_delta_p <= 0.05, 'Selisih Regresi Vertical Velocity vs Nino34 - Δslope = P2 - P1', 'Selisih slope regresi', 'reg_verticalvelocity_simultaneous_delta.png'),
]

for da, sig_mask, title, cbar_label, filename in plot_specs:
    fig, ax = plt.subplots(figsize=(9.5, 6.6))
    mesh = plot_cross_section(ax, da, title, cbar_label, sig_mask=sig_mask)
    cbar = fig.colorbar(mesh, ax=ax, pad=0.04, shrink=0.92, ticks=reg_ticks)
    cbar.set_label(cbar_label, fontsize=14)
    cbar.ax.tick_params(labelsize=12)
    fig.tight_layout()
    fig.savefig(RESULTS_DIR / filename, dpi=300, bbox_inches='tight')
    plt.show()


## 5. Plot DJF-anchored regression

In [ ]:
plot_specs = [
    (reg_djf_p1, reg_djf_p1_p <= 0.05, 'Regresi Vertical Velocity vs Nino34 DJF - P1 1981-2006', 'Slope regresi', 'reg_verticalvelocity_djf_p1.png'),
    (reg_djf_p2, reg_djf_p2_p <= 0.05, 'Regresi Vertical Velocity vs Nino34 DJF - P2 2007-2025', 'Slope regresi', 'reg_verticalvelocity_djf_p2.png'),
    (reg_djf_delta, reg_djf_delta_p <= 0.05, 'Selisih Regresi Vertical Velocity vs Nino34 DJF - Δslope = P2 - P1', 'Selisih slope regresi', 'reg_verticalvelocity_djf_delta.png'),
]

for da, sig_mask, title, cbar_label, filename in plot_specs:
    fig, ax = plt.subplots(figsize=(9.5, 6.6))
    mesh = plot_cross_section(ax, da, title, cbar_label, sig_mask=sig_mask)
    cbar = fig.colorbar(mesh, ax=ax, pad=0.04, shrink=0.92, ticks=reg_ticks)
    cbar.set_label(cbar_label, fontsize=14)
    cbar.ax.tick_params(labelsize=12)
    fig.tight_layout()
    fig.savefig(RESULTS_DIR / filename, dpi=300, bbox_inches='tight')
    plt.show()
